# 1.2 AI-Assisted Coding with Codex

## Course 3: Advanced Classification Models for Student Success

## Introduction

### What Is Codex?

**Codex** is OpenAI's command-line AI coding agent. It runs in your **terminal**, not in a chat window. Once installed and authenticated, you launch it inside any project directory and it works *with the files on your computer* — reading your source code, editing files in place, running shell commands, and executing scripts or notebooks against your real environment.

Codex can:

- **Read and edit files** in your project directory (Python scripts, notebooks, CSVs, configs, etc.)
- **Run shell commands** — install packages with `pip`, run `pytest`, execute `python script.py`, run git operations
- **Iterate on errors** — when a command fails, Codex sees the traceback and tries again
- **Operate across many files** — refactor a function and update every caller, or scaffold a new module
- **Use your real environment** — your installed Python, your actual data, your existing virtualenv

### How Codex Differs from a Chat Tool

Tools like ChatGPT's chat interface (with file uploads / Advanced Data Analysis) run code in a **disposable cloud sandbox** — you upload a CSV, get a response, and the session evaporates. Codex is fundamentally different:

| | Codex (CLI agent) | ChatGPT chat with file upload |
|:--|:------------------|:------------------------------|
| **Where it runs** | Your terminal, on your machine | OpenAI's web sandbox |
| **What it touches** | Your real project files | A copy you uploaded |
| **Persistence** | Edits stay on disk; commit with git | Sandbox is wiped when chat ends |
| **Multi-file work** | Yes — reads/edits across the whole repo | No — limited to what you uploaded |
| **Runs commands** | Yes — any shell command in your project | Limited to its sandbox |

If you've used **Claude Code**, Codex is conceptually very similar — same model of "AI agent in your terminal that operates on your real project." The two are direct competitors.

### Why Learn This Now?

In notebook 1.1 you used **Gemini in Google Colab** — an AI assistant living *inside* a single notebook in the cloud. Codex is the opposite: it lives in your terminal and works across your **entire project on your local machine** (or a dev container). It's the right tool when your work spans multiple files, when you want changes saved to disk and versioned with git, and when you want to keep a normal local Python workflow.

### What Is "Vibecoding"?

**Vibecoding** = using natural language to direct an AI tool to write, run, and iterate on code. Instead of writing every line yourself, you:

1. **Describe** what you want in plain English
2. **Review** the diffs and commands the agent proposes
3. **Iterate** by refining your instructions
4. **Validate** the results against your domain knowledge

> *Think of it as pair programming — you bring the institutional knowledge, Codex brings the coding speed and the willingness to slog through boilerplate.*

### Learning Objectives

1. Understand what Codex is and how it differs from Colab + Gemini
2. Install Codex and run it inside a project directory
3. Learn effective prompting strategies for data science tasks
4. Practice the prompt → review → iterate workflow against real files
5. Apply vibecoding to extend analyses from this course

## 1. How to Access Codex

### Installation

Codex is distributed as a CLI tool. Install it via npm (or Homebrew on macOS):

```bash
# npm (cross-platform)
npm install -g @openai/codex

# Homebrew (macOS)
brew install codex
```

You'll need a ChatGPT Plus, Pro, Team, or Enterprise account (or an OpenAI API key) to authenticate.

### First Run

Open a terminal, navigate to a project directory, and launch Codex:

```bash
cd ~/projects/my-ir-analysis
codex
```

On first launch Codex prompts you to sign in with your ChatGPT account (or paste an API key). After that, you're at the Codex prompt — a chat-like interface inside your terminal where you describe what you want and Codex proposes file edits and shell commands.

### What You'll See

```
┌─────────────────────────────────────────────────────────────┐
│  ~/projects/my-ir-analysis  (main)                          │
│  $ codex                                                     │
│                                                              │
│  ▌                                                           │
│  ▌  Codex is ready. What would you like to do?              │
│  ▌                                                           │
│  ▌  > Build a logistic regression on student_academics_data │
│  ▌    .csv to predict DEPARTED, save the script as          │
│  ▌    models/logreg.py, and write metrics to a markdown     │
│  ▌    report.                                               │
│  ▌                                                           │
│  ▌  [Codex proposes: create models/logreg.py]               │
│  ▌  [Codex proposes: run `python models/logreg.py`]         │
│  ▌  Approve? [y/n/edit]                                     │
└─────────────────────────────────────────────────────────────┘
```

Codex shows you each proposed file edit (as a diff) and each proposed shell command before running it. You approve, edit, or reject. By default, destructive or network-touching commands require explicit approval.

### The Codex Workflow

```
┌──────────────────────────────────────────────────────────┐
│  1. cd into your project, run `codex`                     │
│  2. Describe a task in natural language                   │
│  3. Codex reads relevant files, proposes edits & commands │
│  4. You review the diff / command, approve or refine      │
│  5. Codex applies edits, runs commands, reads outputs     │
│  6. Iterate via follow-up messages — full project context │
│     persists across the session                           │
│  7. Commit the changes with git                           │
└──────────────────────────────────────────────────────────┘
```

### Approval Modes

Codex has different autonomy levels you can configure:

- **Suggest** (default) — Codex proposes every edit and command; you approve each one
- **Auto-edit** — file edits are applied automatically; commands still need approval
- **Full-auto** — both edits and commands run automatically (use only on disposable branches or in containers)

Start in *Suggest* mode while you're learning what Codex tends to do, then loosen the leash once you trust the workflow.

## 2. Prompting Strategies for Data Science

Because Codex acts on your real project, the best prompts name **specific files**, **specific paths**, and **specific outputs you want on disk**. Vague prompts like "analyze my data" leave too much room for Codex to guess where things live.

### The CRISP Prompting Framework

| Letter | Meaning | Example for Codex |
|:-------|:--------|:------------------|
| **C** | Context | "We have `data/student_academics_data.csv` with student records and a `SEM_3_STATUS` column..." |
| **R** | Role | "Act as an institutional research analyst..." |
| **I** | Instructions | "Create a script `models/logreg.py` that builds a logistic regression to predict departure..." |
| **S** | Specifics | "Use L2 regularization with C=0.1, scale features with StandardScaler, random_state=42..." |
| **P** | Product | "Save metrics to `reports/logreg_metrics.json` and a ROC curve to `reports/logreg_roc.png`" |

### Example Prompts (typed at the Codex prompt in your terminal)

#### Prompt 1: Exploratory Data Analysis

```
Read data/student_academics_data.csv. Create a notebook
notebooks/eda.ipynb that:
1. Shows summary statistics for HS_GPA, UNITS_ATTEMPTED_1,
   UNITS_COMPLETED_1, GPA_1, DFW_RATE_1, and DEPARTED
2. Plots the distribution of each numeric variable
3. Shows a correlation heatmap
4. Prints the top 3 features most correlated with DEPARTED

Run the notebook end-to-end and confirm all cells execute cleanly.
```

#### Prompt 2: Build a Classification Model

```
Create models/logreg.py that:
- Loads data/student_academics_data.csv
- Treats DEPARTED as the target
- Uses all numeric columns except DEPARTED as features
- Fits a logistic regression with L2 penalty, C=0.1, on a
  StandardScaler-transformed 80/20 train/test split (random_state=42)
- Prints AUC, F1, precision, recall on the test set
- Writes the metrics to reports/logreg_metrics.json
- Saves the ROC curve to reports/logreg_roc.png
- Saves the top 10 absolute coefficients to reports/logreg_coefs.csv

Then run the script and show me the printed metrics.
```

#### Prompt 3: Compare Models

```
Add models/rf.py and models/xgb.py that mirror models/logreg.py
but use RandomForestClassifier(n_estimators=200) and
XGBClassifier(learning_rate=0.1) respectively. Then create
reports/compare_models.py that loads the three metrics JSON files,
prints a comparison table (AUC, F1, Precision, Recall, Accuracy),
and saves an overlay ROC plot to reports/compare_roc.png.
```

### Why This Style Works

Notice every prompt names **files Codex should create**, **paths it should read**, and **commands it should run**. That gives Codex something concrete to do and gives *you* something concrete to review (a diff and an output file). Compare to "build me a model and show me how it performed" — that style works for chat tools but wastes Codex's strengths.

### Iteration Pattern

Codex shines at **follow-ups**. After the first prompt lands, you can say things like:

- *"Now add 5-fold cross-validation in models/logreg.py and update the metrics JSON."*
- *"The ROC plot is using the default colors — make logistic blue, RF orange, XGBoost green, and add a thicker chance-line."*
- *"Refactor the three model scripts to share a common `data_prep.py` module."*

Codex keeps full session context, so it knows what files exist and what your previous instructions were.

## 3. Best Practices

### DO ✓

- **Work in a git repo.** Codex edits files in place; git is your safety net (`git diff`, `git restore`, branches).
- **Review every diff** before approving — Codex sometimes over-edits or touches unrelated files.
- **Be specific** about file paths, function names, metric names, and chart styles.
- **Commit often.** Treat each clean Codex turn like a small commit so you can roll back specific changes.
- **Tell Codex about your environment** — Python version, virtualenv, key dependencies — when it matters.
- **Use a `CLAUDE.md` / `AGENTS.md` / `CODEX.md`-style project notes file** in the repo to give Codex persistent context (data dictionaries, conventions, what *not* to touch).
- **Run things yourself** when in doubt — Codex can run commands, but you can always run them in your own terminal too.

### DON'T ✗

- **Don't run full-auto on `main`.** Use a feature branch (or a worktree / container) so an over-eager edit can be discarded.
- **Don't approve commands you don't understand** — especially anything involving `rm -rf`, `git push --force`, package uninstalls, or network calls to unfamiliar URLs.
- **Don't share sensitive data without checking governance.** Even though Codex runs locally, your prompts and file *contents* are sent to OpenAI's models.
- **Don't ignore errors.** If a command fails, ask Codex to diagnose — don't just retry blindly.
- **Don't skip validation.** Numerical results can look reasonable and still be wrong (wrong target, leaked features, mis-scaled inputs). Always sanity-check.

### The 80/20 Rule of Vibecoding

> Codex gets you 80% of the way there in 20% of the time.
> Your job is the remaining 20% — validation, interpretation, and institutional context.

## 4. Mapping Course Content to Codex Prompts

Here's how each module in this course maps to a Codex prompt. Each one tells Codex what to create, where to save it, and what to run.

| Module | Task | Codex Prompt Starter |
|:-------|:-----|:--------------------|
| 2. Regularization | L1/L2 Logistic | "Add `models/logreg_l2.py` that fits a logistic regression with L2 penalty C=0.1 on `data/student_academics_data.csv` and writes metrics to `reports/`." |
| 3. Tree-Based | Random Forest | "Add `models/rf.py` with 200 trees, max_depth=12, and save feature importances to `reports/rf_importance.csv`." |
| 3. Tree-Based | XGBoost | "Add `models/xgb.py` with learning_rate=0.1 and write the ROC plot to `reports/xgb_roc.png`." |
| 4. Comparison | Head-to-head | "Create `reports/compare.py` that loads each model's metrics JSON and prints a head-to-head AUC/F1/Precision/Recall table." |
| 5. Unsupervised | K-Means + PCA | "Add `analysis/clusters.py` that runs K-Means k=4 with PCA(2) projection and saves the scatter plot to `reports/clusters.png`." |

### Practice Exercise

Set up a small project directory and run Codex against it.

```bash
mkdir -p ~/codex-practice/{data,models,reports}
cd ~/codex-practice
git init
# copy training.csv into data/
codex
```

Then at the Codex prompt, type:

```
We have data/training.csv. The column SEM_3_STATUS marks 'E' = enrolled
(student returned), and any other value means departed.

Please:
1. Create models/logreg.py that:
   - Loads data/training.csv
   - Adds a binary DEPARTED column (1 if SEM_3_STATUS != 'E', else 0)
   - Selects features: HS_GPA, UNITS_ATTEMPTED_1, UNITS_COMPLETED_1,
     DFW_UNITS_1, GPA_1, DFW_RATE_1
   - Fits a logistic regression with L2, C=0.1, on a StandardScaler-
     transformed 80/20 split (random_state=42)
   - Prints AUC, F1, precision, recall
   - Writes the top coefficients (sorted by |value|) to
     reports/logreg_coefs.csv
   - Saves the ROC curve to reports/logreg_roc.png
2. Run the script and show me the printed metrics.
```

After Codex finishes, inspect the new files yourself with `ls reports/`, open `models/logreg.py` in your editor, and run `git diff` to see exactly what was created.

## Summary

### Key Takeaways

1. **Codex is a CLI agent** that runs in your terminal and operates on your local project — not a chat sandbox.
2. It **reads, edits, and runs** real files and commands. Your safety net is **git** + the approval prompt before each action.
3. **Vibecoding** = describe → review the diff → iterate → validate.
4. **Good Codex prompts** name specific files, specific paths, and specific outputs you want produced on disk.
5. **You are still the analyst.** Codex is the coding accelerator; you bring domain expertise and validate every result.
6. Conceptually, Codex sits in the same category as **Claude Code** — terminal-based agents that work on your real repo.

### What's Next

Now you've seen both Colab + Gemini (notebook 1.1) and Codex (this notebook). They're very different tools — one lives in a single cloud notebook, the other lives in your terminal across your whole project. In **notebook 1.3**, we compare them side by side so you can pick the right one for each job, or use both.

**Proceed to:** `1.3 Choosing Your AI Coding Tool`